# Top 10 Conditions, Procedures, and Medications

Analysis of the Texas Workers' Compensation data using OMOP CDM tables.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect('/workspaces/txwc/tx_workers_comp.db', read_only=True)

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Top 10 Conditions

From the OMOP condition_occurrence table.

In [3]:
top_conditions = con.execute("""
    select 
        co.condition_source_value as source_code,
        c.concept_name as condition_name,
        count(*) as occurrence_count
    from omop.condition_occurrence co
    left join omop.concept c 
        on co.condition_concept_id = c.concept_id
    where co.condition_source_value is not null
    group by co.condition_source_value, c.concept_name
    order by occurrence_count desc
    limit 10
""").fetchdf()

print("Top 10 Conditions")
print("=" * 80)
top_conditions

Top 10 Conditions


,source_code,condition_name,occurrence_count
0,847.2,Lumbar sprain,2233616
1,724.2,Low back pain,1226121
2,847.0,Neck sprain,1021180
3,724.4,Peripheral neuritis,935635
4,840.9,Dislocations/sprains/strains,934531
5,722.10,Displacement of lumbar intervertebral disc without myelopathy,928007
6,S39.012A,"Injury of muscle and tendon of abdomen, lower back and pelvis",918443
7,S33.5XXD,Lumbar sprain,807208
8,719.41,Shoulder joint pain,741725
9,844.9,Soft tissue injury,702301


## Top 10 Procedures

From the OMOP procedure_occurrence table.

In [4]:
top_procedures = con.execute("""
    select 
        po.procedure_source_value as source_code,
        c.concept_name as procedure_name,
        count(*) as occurrence_count
    from omop.procedure_occurrence po
    left join omop.concept c 
        on po.procedure_concept_id = c.concept_id
    where po.procedure_source_value is not null
    group by po.procedure_source_value, c.concept_name
    order by occurrence_count desc
    limit 10
""").fetchdf()

print("Top 10 Procedures")
print("=" * 80)
top_procedures

Top 10 Procedures


,source_code,procedure_name,occurrence_count
0,G0283,"Electrical stimulation (unattended), to one or more areas for indication(s) other than wound care, as part of a therapy plan of care",1311879
1,99.99,Other miscellaneous procedures,350059
2,J3490,Unclassified drugs,100350
3,J8499,"Prescription drug, oral, non chemotherapeutic, nos",71430
4,G0299,"Direct skilled nursing services of a registered nurse (rn) in the home health or hospice setting, each 15 minutes",60649
5,S9124,"Nursing care, in the home; by licensed practical nurse, per hour",48978
6,J3590,Unclassified biologics,47378
7,S9123,"Nursing care, in the home; by registered nurse, per hour (use for general nursing care only, not to be used when cpt codes 99500-99602 can be used)",44500
8,J7799,"Noc drugs, other than inhalation drugs, administered through dme",37314
9,J7999,"Compounded drug, not otherwise classified",16532


## Top 10 Medications (by Drug Product)

From the OMOP drug_exposure table.

In [5]:
top_drugs = con.execute("""
    select 
        de.drug_source_value as ndc_code,
        c.concept_name as drug_name,
        count(*) as rx_count
    from omop.drug_exposure de
    left join omop.concept c 
        on de.drug_concept_id = c.concept_id
    where de.drug_source_value is not null
    group by de.drug_source_value, c.concept_name
    order by rx_count desc
    limit 10
""").fetchdf()

print("Top 10 Drug Products")
print("=" * 80)
top_drugs

Top 10 Drug Products


,ndc_code,drug_name,rx_count
0,J1885,ketorolac Injectable Solution [Toradol],358078
1,99999999999,No matching concept,275878
2,65162062711,tramadol hydrochloride 50 MG Oral Tablet,257623
3,J2405,ondansetron Injection,230114
4,J1100,dexamethasone Injection,210224
5,J3010,fentanyl 0.05 MG/ML Injection [Sublimaze],209299
6,53746046605,ibuprofen 800 MG Oral Tablet,179690
7,J2250,midazolam Injectable Solution [Versed],177339
8,J0690,cefazolin Injection,175098
9,00591085305,acetaminophen 325 MG / hydrocodone bitartrate 10 MG Oral Tablet,151498


## Top 10 Medications by Ingredient

Rolling up drug exposures to their RxNorm ingredient using concept_ancestor.

In [6]:
top_ingredients = con.execute("""
    select 
        ingredient.concept_name as ingredient_name,
        count(*) as rx_count
    from omop.drug_exposure de
    join omop.concept_ancestor ca 
        on de.drug_concept_id = ca.descendant_concept_id
    join omop.concept ingredient 
        on ca.ancestor_concept_id = ingredient.concept_id
    where ingredient.vocabulary_id = 'RxNorm'
      and ingredient.concept_class_id = 'Ingredient'
      and de.drug_concept_id is not null
      and de.drug_concept_id != 0
    group by ingredient.concept_name
    order by rx_count desc
    limit 10
""").fetchdf()

print("Top 10 Medication Ingredients")
print("=" * 80)
top_ingredients

Top 10 Medication Ingredients


,ingredient_name,rx_count
0,acetaminophen,2735379
1,hydrocodone,2052047
2,tramadol,940839
3,cyclobenzaprine,929015
4,ibuprofen,780152
5,gabapentin,748233
6,naproxen,665036
7,meloxicam,501974
8,pregabalin,404941
9,ketorolac,369808


## Top 10 Drug Classes by Established Pharmacologic Class (EPC)

Using the rxclass reference table to analyze drug exposures by FDA Established Pharmacologic Class.

In [7]:
top_epc = con.execute("""
    select 
        rc.className as drug_class,
        count(*) as rx_count
    from omop.drug_exposure de
    join omop.concept c 
        on de.drug_concept_id = c.concept_id
    join reference_data.rxclass rc 
        on c.concept_code = rc.rxcui
    where de.drug_concept_id is not null
      and de.drug_concept_id != 0
      and rc.classType = 'EPC'
    group by rc.className
    order by rx_count desc
    limit 10
""").fetchdf()

print("Top 10 Established Pharmacologic Classes (EPC)")
print("=" * 80)
top_epc

Top 10 Established Pharmacologic Classes (EPC)


,drug_class,rx_count
0,Opioid Agonist,6852984
1,Nonsteroidal Anti-inflammatory Drug,5467660
2,Muscle Relaxant,2817018
3,Corticosteroid,920426
4,Central alpha-2 Adrenergic Agonist,709478
5,Serotonin and Norepinephrine Reuptake Inhibitor,658322
6,Tricyclic Antidepressant,465730
7,gamma-Aminobutyric Acid A Receptor Positive Modulator,391790
8,Benzodiazepine,326216
9,Serotonin Reuptake Inhibitor,318958


## Top 10 Drug Classes by Disease Indication

Using the rxclass reference table to analyze drug exposures by disease-based classification.

In [8]:
top_disease = con.execute("""
    select 
        rc.className as disease_indication,
        count(*) as rx_count
    from omop.drug_exposure de
    join omop.concept c 
        on de.drug_concept_id = c.concept_id
    join reference_data.rxclass rc 
        on c.concept_code = rc.rxcui
    where de.drug_concept_id is not null
      and de.drug_concept_id != 0
      and rc.classType = 'DISEASE'
    group by rc.className
    order by rx_count desc
    limit 10
""").fetchdf()

print("Top 10 Disease Indications")
print("=" * 80)
top_disease

Top 10 Disease Indications


,disease_indication,rx_count
0,Drug Hypersensitivity,12727872
1,Pain,10023965
2,Fever,6808201
3,Asthma,4245702
4,Liver Diseases,3169662
5,"Arthritis, Rheumatoid",3082518
6,"Pain, Postoperative",3042634
7,Intestinal Obstruction,2950402
8,Respiratory Insufficiency,2822661
9,Osteoarthritis,2722828


## Summary Statistics

In [9]:
stats = con.execute("""
    select 'Conditions' as domain, count(*) as record_count from omop.condition_occurrence
    union all
    select 'Procedures', count(*) from omop.procedure_occurrence
    union all
    select 'Drug Exposures', count(*) from omop.drug_exposure
    union all
    select 'Measurements', count(*) from omop.measurement
    union all
    select 'Observations', count(*) from omop.observation
    union all
    select 'Visits', count(*) from omop.visit_occurrence
    union all
    select 'Persons', count(*) from omop.person
""").fetchdf()

print("OMOP Table Record Counts")
print("=" * 40)
stats

OMOP Table Record Counts


,domain,record_count
0,Conditions,84947725
1,Procedures,2605751
2,Drug Exposures,16789640
3,Measurements,1231275
4,Observations,10560423
5,Visits,50548780
6,Persons,24002149


In [10]:
con.close()